In [1]:
%%configure -f
{
  "conf": {
    "spark.master": "yarn",
    "spark.speculation": "false",

    "spark.driver.cores": "8",
    "spark.driver.memory": "384g",
    "spark.driver.memoryOverhead": "96g",

    "spark.yarn.am.cores": "6",
    "spark.yarn.am.memory": "384g",
    "spark.yarn.am.memoryOverhead": "96g",

    "spark.driver.maxResultSize": "20g",

    "spark.executor.cores": "4",
    "spark.executor.memory": "18g",
    "spark.executor.memoryOverhead": "8g",

    "spark.dynamicAllocation.enabled": "true",
    "spark.dynamicAllocation.minExecutors": "189",
    "spark.dynamicAllocation.initialExecutors": "189",
    "spark.dynamicAllocation.maxExecutors": "1000",
    "spark.dynamicAllocation.executorIdleTimeout": "60s",
    "spark.dynamicAllocation.cachedExecutorIdleTimeout": "300s",
    "spark.dynamicAllocation.schedulerBacklogTimeout": "1s",
    "spark.dynamicAllocation.sustainedSchedulerBacklogTimeout": "1s",
    "spark.dynamicAllocation.executorAllocationRatio": "1.0",

    "spark.locality.wait": "1s",

    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "false",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "268435456",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.localShuffleReader.enabled": "true",
    "spark.sql.shuffle.partitions": "1024",
    "spark.default.parallelism": "1024",

    "spark.network.timeout": "800s",
    "spark.executor.heartbeatInterval": "60s",
    "spark.kryoserializer.buffer.max": "1g",
    "spark.rpc.message.maxSize": "1024",

    "spark.hadoop.fs.s3a.aws.credentials.provider": "com.amazonaws.auth.DefaultAWSCredentialsProviderChain",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryo.registrator": "is.hail.kryo.HailKryoRegistrator",

    "spark.hadoop.fs.s3.maxConnections": "50000",
    "spark.hadoop.fs.s3.connection.timeout": "120000",
    "spark.hadoop.fs.s3.socket.timeout": "120000",
    "spark.hadoop.fs.s3.maxRetries": "20",
    "spark.hadoop.fs.s3a.threads.max": "256",
    "spark.hadoop.fs.s3a.connection.maximum": "50000",
    "spark.hadoop.fs.s3a.connection.timeout": "120000",
    "spark.hadoop.fs.s3a.socket.timeout": "120000",
    "spark.hadoop.fs.s3a.attempts.maximum": "20",
    "spark.hadoop.fs.s3a.retry.interval": "1000",
    "spark.hadoop.fs.s3a.fast.upload": "true",
    "spark.hadoop.fs.s3a.multipart.size": "104857600",
    "spark.hadoop.fs.s3a.threads.keepalivetime": "60000",
    "spark.hadoop.fs.s3a.connection.establish.timeout": "30000",
    "spark.hadoop.fs.s3a.multipart.purge.age": "86400000"
  },
  "executorCores": 4,
  "executorMemory": "18G",
  "driverCores": 8,
  "driverMemory": "384G"
}

In [2]:
# Initialize Hail against the running SparkContext from Livy/EMR Notebooks
import hail as hl
hl.init(sc, log="/tmp/hail.log")

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1781578145249_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/usr/local/lib/python3.11/site-packages/hail/backend/spark_backend.py:76: UserWarning: Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jar
  warnings.warn(
Running on Apache Spark version 3.5.5-amzn-1
SparkUI available at http://ip-192-168-78-46.ap-southeast-1.compute.internal:41141
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.138-58956ebc28fc
LOGGING: writing to /tmp/hail.log

In [3]:
# source
vds_prefix = "s3://precise-scratch/goypav/SG10K_Health/VDS/"
annotated_vds_prefix = vds_prefix + "step1/annotated_vds/"
vds_step2_prefix = vds_prefix + "step2/"

# input
manifest_uri = "s3://precise-scratch/goypav/SG10K_Health/VDS/vds_step1_manifest_SG10K_Health_first1000_20260615.txt"

# output



FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# -----------------------------
# 1. Read manifest of VDS paths
# -----------------------------
# aws s3 ls s3://precise-scratch/goypav/1KG/test_r4.x/VDS/step1/annotated_vds/ \
# | awk '$NF ~ /^batch_20260522.*\.vds\/?$/ {
#     print "s3://precise-scratch/goypav/1KG/test_r4.x/VDS/step1/annotated_vds/" $NF
# }' \
# | sort \
# > vds_step1_manifest.txt
# copy to s3
# aws s3 cp vds_step1_manifest.txt s3://precise-scratch/goypav/1KG/test_r4.x/VDS/ --dryrun

with hl.current_backend().fs.open(manifest_uri) as f:
    vds_paths = [line.strip() for line in f if line.strip()]

print(f"Found {len(vds_paths):,} VDSes")
print(vds_paths[:5])

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Found 1,000 VDSes
['s3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260615_161224_0000.vds/', 's3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260615_161224_0001.vds/', 's3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260615_161224_0002.vds/', 's3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260615_161225_0003.vds/', 's3://precise-scratch/goypav/SG10K_Health/VDS/step1/annotated_vds/batch_20260615_161225_0004.vds/']

In [5]:
# -----------------------------
# 2. Define batching + output paths
# -----------------------------

batch_size = 100

logs_prefix = vds_step2_prefix + "logs/"

vds_batches = [
    vds_paths[i:i + batch_size]
    for i in range(0, len(vds_paths), batch_size)
]

print(f"Number of batches: {len(vds_batches)}")
print(f"Logs will be written to: {logs_prefix}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Number of batches: 10
Logs will be written to: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/logs/

In [6]:
# -----------------------------
# 3. Combine VDSes in batches of 100
# safer loop: no retained Hail objects
# -----------------------------

import time
import traceback
import gc
from datetime import datetime

fs = hl.current_backend().fs

def exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False

for batch_idx, batch_paths in enumerate(vds_batches):
    batch_name = f"batch_{batch_idx:04d}"
    out_uri = f"{vds_step2_prefix}{batch_name}.vds"
    log_uri = f"{logs_prefix}{batch_name}.log"
    input_manifest_uri = f"{logs_prefix}{batch_name}.inputs.txt"

    start = time.time()
    log_lines = []

    def log(msg):
        line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
        print(line)
        log_lines.append(line)

    try:
        log(f"Starting {batch_name}")
        log(f"Number of input VDSes: {len(batch_paths)}")
        log(f"Output URI: {out_uri}")

        with fs.open(input_manifest_uri, "w") as f:
            f.write("\n".join(batch_paths) + "\n")

        if exists(out_uri):
            log("Output already exists, skipping")
            continue

        # important: build inside loop, then delete after write
        input_vdses = [hl.vds.read_vds(p) for p in batch_paths]

        combined_vds = hl.vds.combiner.combine.combine_variant_datasets(input_vdses)

        combined_vds.write(out_uri, overwrite=False)

        elapsed = time.time() - start
        log(f"Completed {batch_name}")
        log(f"Runtime: {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)")

    except Exception as e:
        elapsed = time.time() - start
        log(f"FAILED {batch_name}")
        log(f"Runtime before failure: {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)")
        log(f"Error: {repr(e)}")
        log(traceback.format_exc())
        raise

    finally:
        with fs.open(log_uri, "w") as f:
            f.write("\n".join(log_lines) + "\n")

        # critical cleanup between batches
        for name in ["input_vdses", "combined_vds"]:
            if name in locals():
                del locals()[name]

        gc.collect()

## DOES NOT WORK FOR 74 parts VDS

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[2026-06-16 03:04:01] Starting batch_0000
[2026-06-16 03:04:01] Number of input VDSes: 100
[2026-06-16 03:04:01] Output URI: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch_0000.vds
9700
[2026-06-16 03:45:06] Completed batch_0000
[2026-06-16 03:45:06] Runtime: 2464.9 seconds (41.08 minutes)
296
82914
[2026-06-16 03:45:06] Starting batch_0001
[2026-06-16 03:45:06] Number of input VDSes: 100
[2026-06-16 03:45:06] Output URI: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch_0001.vds
9700
[2026-06-16 04:27:29] Completed batch_0001
[2026-06-16 04:27:29] Runtime: 2543.2 seconds (42.39 minutes)
296
87449
[2026-06-16 04:27:30] Starting batch_0002
[2026-06-16 04:27:30] Number of input VDSes: 100
[2026-06-16 04:27:30] Output URI: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch_0002.vds
9700
[2026-06-16 05:06:29] Completed batch_0002
[2026-06-16 05:06:29] Runtime: 2339.7 seconds (39.00 minutes)
296
87449
[2026-06-16 05:06:30] Starting batch_0003
[2026-06-16 05:06:30] Num

## check output

In [7]:
# -----------------------------
# Read first combined batch VDS
# -----------------------------

vds_batch1_uri = f"{vds_step2_prefix}batch_0000.vds"

vds = hl.vds.read_vds(vds_batch1_uri)

print(f"Loaded: {vds_batch1_uri}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Loaded: s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch_0000.vds

In [8]:
# -----------------------------
# Inspect schemas
# -----------------------------

print("=== Reference data ===")
vds.reference_data.describe()


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Reference data ===
----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LEN': int32
    'gvcf_qual': float64
    'gvcf_filters': set<str>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [9]:
print("\n=== Variant data ===")
vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


=== Variant data ===
----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64>
    'DP': int32
    'F1R2': array<int32>
    'F2R1': array<int32>
    'GP': array<float64>
    'GQ': int32
    'ICNT': array<int32>
    'MB': array<int32>
    'MIN_DP': int32
    'PRI': array<float64>
    'PS': int32
    'SB': array<int32>


In [10]:
# -----------------------------
# Basic summary
# -----------------------------

print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples():,}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")
print(f"Variant partitions: {vds.variant_data.n_partitions():,}")
print(f"Reference partitions: {vds.reference_data.n_partitions():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reference genome: GRCh38
Number of samples: 100
Total number of variants: 37,755,085
Variant partitions: 632
Reference partitions: 632